# PoGS geodesic distance (Potential-minimizing Geodesic Search, $\lambda=0$)

PoGS (PepCompass, Mozejko et al.) joins a parent $z_0=z$ to a target $z_N=z'$ by a discrete latent path
$Z=(z_0,\dots,z_N)$ whose interior way-points are found by **ADAM**-minimising the energy

$$E_{\lambda,\mu}(Z)=\underbrace{\sum_{k=0}^{N-1}\lVert X_{k+1}-X_k\rVert_2^2}_{\text{kinetic: ambient geometric similarity}}
+\;\lambda\underbrace{\sum_{k=0}^{N}\Phi(X_k)}_{\text{potential: property}}
+\;\mu\underbrace{\sum_{k=0}^{N-1}\lVert z_{k+1}-z_k\rVert_2^2}_{\text{latent regulariser (Euclidean)}},\qquad X_k=\mathrm{Dec}(z_k).$$

Every norm is **plain Euclidean**: the kinetic term is the *proper chord distance in ambient (soft-peptide) space*, the regulariser is the Euclidean latent step — the pull-back metric $G=J^\top J$ is never formed. The reported PoGS **distance** is the ambient chord *length* of the optimised path,

$$d_{\mathrm{PoGS}}^{(\lambda)}(z,z')=\sum_{k=0}^{N-1}\big\lVert \mathrm{Dec}(z_k^*)-\mathrm{Dec}(z_{k+1}^*)\big\rVert_2,\qquad \{z_k^*\}=\arg\min E_{\lambda,\mu}.$$

With $\lambda=0$ the potential drops out and $d_{\mathrm{PoGS}}^{(0)}$ is the ($\mu$-regularised) **geodesic distance**. The implementation lives in `analysis/scripts/thesis_figures/_pogs.py`.

*Note:* `lr=1e-3` matters — larger ADAM rates overshoot and push way-points into saturated decoder regions, so the realised length grows instead of shrinking.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr

ROOT = next(p for p in Path.cwd().resolve().parents if p.name == "pep-compass")
sys.path.insert(0, str(ROOT / "analysis" / "scripts" / "thesis_figures"))
sys.path.insert(0, str(ROOT / "src"))
from _common import CACHE  # noqa: E402
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import HydrAMPEncoderDecoder  # noqa: E402
from _pogs import pogs_distance_to_parent, direct_chord_to_parent  # noqa: E402

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hyd = HydrAMPEncoderDecoder(jacobian_mode="approx", jacobian_eps=1e-6, field_eps=1e-6, device=DEV)
hyd.eval()
DEV

## Sanity demo: $d_{\mathrm{PoGS}}^{(0)}$ vs the graph geodesic and the direct ambient chord

Using the cached RQ3 candidate set, the $\lambda=0$ PoGS length should (a) sit at or just above the direct chord $\lVert\mathrm{Dec}(z')-\mathrm{Dec}(z)\rVert$, (b) grow monotonically with the mutation count, and (c) correlate with the cached distances.

In [ ]:
df = pd.read_parquet(CACHE / "_thesis_rq5_distances.parquet").reset_index(drop=True)
peps = list(dict.fromkeys(df["pep"]))[:3]
rows = []
for pep in peps:
    g = df[df.pep == pep].head(60).reset_index(drop=True)
    with torch.no_grad():
        z0 = hyd.encode_peptides([pep]).detach()[0]
        zz = hyd.encode_peptides(g["seq"].tolist()).detach()
    d_pogs = pogs_distance_to_parent(hyd, z0, zz, lam=0.0)  # n_seg=8, mu=1e-2, lr=1e-3, 500 steps
    d_chord = direct_chord_to_parent(hyd, z0, zz)
    for i in range(len(g)):
        rows.append(dict(pep=pep, n_mut=int(g.n_mut[i]), pogs=d_pogs[i], chord=d_chord[i],
                         dist_geo=g.dist_geo[i], dist_eucl=g.dist_eucl[i]))
r = pd.DataFrame(rows)
print("PoGS length >= direct chord:", f"{(r.pogs >= r.chord - 1e-6).mean()*100:.0f}%")
print("Spearman(pogs, chord)    =", round(spearmanr(r.pogs, r.chord).correlation, 3))
print("Spearman(pogs, dist_geo) =", round(spearmanr(r.pogs, r.dist_geo).correlation, 3))
print("Spearman(pogs, dist_eucl)=", round(spearmanr(r.pogs, r.dist_eucl).correlation, 3))
r.groupby("n_mut")[["pogs", "chord", "dist_geo"]].median()

The full caches (`dist_pogs` column on the 385- and 845-peptide candidate sets) are produced on Bury by `analysis/scripts/thesis_figures/_exp_pogs_distance_cache.py` (SLURM wrapper `scripts/pogs_distance_gpu.sh`). The TANDEM-A and MUTANG+ alignment checks under $d_{\mathrm{PoGS}}$ are in `fig_dist_pogs_tandem.py` / `fig_rq5_mutangplus_pogs.py`.

Setting $\lambda>0$ with a property potential $\Phi$ (passed as the `phi` callable) turns this back into the full Potential-minimizing Geodesic Search.